<a href="https://colab.research.google.com/github/tkoziara/Piper-Tools/blob/main/Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0) Verify Colab GPU and environment

In [ ]:
import torch, sys
print('Python', sys.version.split()[0])
print('PyTorch', torch.__version__)
print('CUDA available', torch.cuda.is_available())
# show GPU info if available
if torch.cuda.is_available():
    try:
        import subprocess
        print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip())
    except Exception as e:
        print('nvidia-smi failed:', e)

## 1) Bootstrap environment (run once)

In [ ]:
# Cell 3 — inspect, strip hyperparams entirely, save weights-only temp ckpt, then run training
import sys, os, torch, pathlib, tempfile, subprocess, textwrap
from pathlib import Path

# ensure piper source is importable
PIPER_SRC = Path('/content/piper1-gpl/src')
if PIPER_SRC.exists():
    sys.path.insert(0, str(PIPER_SRC))

torch.serialization.add_safe_globals([pathlib.PosixPath])

print("DEBUG: VOICE_NAME =", globals().get('VOICE_NAME'))
print("DEBUG: DATA_ROOT  =", globals().get('DATA_ROOT'))
print("DEBUG: CKPT_PATH  =", globals().get('CKPT_PATH'))

DATA_ROOT = Path(DATA_ROOT)
CONFIG_PATH = DATA_ROOT / f"{VOICE_NAME}.json"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing required config: {CONFIG_PATH}")

CKPT_PATH = Path(CKPT_PATH) if 'CKPT_PATH' in globals() and CKPT_PATH is not None else None

use_ckpt = None
tmp_ckpt = None

if CKPT_PATH and CKPT_PATH.exists():
    print("DEBUG: loading checkpoint for weights extraction:", CKPT_PATH)
    ckpt = torch.load(str(CKPT_PATH), map_location='cpu', weights_only=False)
    # Remove any hyperparameter metadata entirely so Lightning won't parse them into CLI args
    if 'hyper_parameters' in ckpt:
        print("DEBUG: removing 'hyper_parameters' from checkpoint")
        ckpt.pop('hyper_parameters', None)
    if 'hparams' in ckpt:
        print("DEBUG: removing 'hparams' from checkpoint")
        ckpt.pop('hparams', None)
    # Optionally keep only state_dict to be extra-safe (weights-only)
    minimal = {k: v for k, v in ckpt.items() if k in ('state_dict', 'optimizer_states', 'lr_schedulers', 'epoch', 'global_step')}
    if not minimal:
        # fallback: save the original without hyperparams
        minimal = ckpt
    # Save minimal/weights-only checkpoint to a temp file
    fd, tmp_path = tempfile.mkstemp(suffix='.ckpt', prefix='weights_only_')
    os.close(fd)
    tmp_ckpt = Path(tmp_path)
    torch.save(minimal, str(tmp_ckpt))
    use_ckpt = tmp_ckpt
    print("DEBUG: saved weights-only temporary checkpoint:", tmp_ckpt)
else:
    print("INFO: no checkpoint present (will start fresh)")

# Build CLI args (ensure --data.config_path provided)
cli_args = [
    'fit',
    '--data.voice_name', VOICE_NAME,
    '--data.csv_path', str(DATA_ROOT / 'metadata.csv'),
    '--data.audio_dir', str(DATA_ROOT / 'wavs'),
    '--data.cache_dir', str(globals().get('CACHE_DIR', '/content/piper_cache')),
    '--data.espeak_voice', globals().get('ESPEAK_VOICE', 'pl'),
    '--data.batch_size', str(globals().get('BATCH_SIZE', 32)),
    '--data.config_path', str(CONFIG_PATH),
    '--model.sample_rate', str(globals().get('SAMPLE_RATE', 22050)),
]

if use_ckpt:
    # Pass the sanitized weights-only checkpoint (no hparams -> no CLI merge)
    cli_args += ['--ckpt_path', str(use_ckpt), '--weights_only', 'true']
else:
    print("DEBUG: not passing --ckpt_path (fresh start)")

if 'MAX_EPOCHS' in globals() and MAX_EPOCHS is not None:
    cli_args += ['--trainer.max_epochs', str(MAX_EPOCHS)]
    print("DEBUG: trainer.max_epochs =", MAX_EPOCHS)

if torch.cuda.is_available():
    cli_args += ['--trainer.accelerator', 'gpu', '--trainer.devices', '1']
    print("DEBUG: requesting GPU")

print("DEBUG: final CLI args:", cli_args)

# launcher that registers safe globals then runs the module
launcher = textwrap.dedent(f\"\"\"\
    import runpy, sys, pathlib, torch
    torch.serialization.add_safe_globals([pathlib.PosixPath])
    sys.argv = ['piper.train'] + {repr(cli_args)}
    runpy.run_module('piper.train', run_name='__main__')
\"\"\")
with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
    f.write(launcher)
    launcher_path = f.name

print("DEBUG: launcher written to", launcher_path)
python_exe = sys.executable
cmd = [python_exe, '-u', launcher_path]
print("DEBUG: launching training subprocess:", ' '.join(cmd))

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                        cwd=str(PIPER_SRC.parent if PIPER_SRC.exists() else os.getcwd()))
try:
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
finally:
    try:
        proc.stdout.close()
    except Exception:
        pass

print("DEBUG: subprocess exited with", ret)

# cleanup
if tmp_ckpt and tmp_ckpt.exists():
    try:
        tmp_ckpt.unlink()
        print("DEBUG: removed temporary ckpt", tmp_ckpt)
    except Exception as e:
        print("WARNING: could not remove temp ckpt:", e)

if ret != 0:
    raise subprocess.CalledProcessError(ret, cmd)

## 2) Mount Drive and set paths/hyperparameters

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
VOICE_NAME = 'tomek_pl'
# Maximum number of training epochs (set to None to leave Trainer default)
MAX_EPOCHS = 20
ESPEAK_VOICE = 'pl'
SAMPLE_RATE = 22050
BATCH_SIZE = 32
# Checkpoint monitor settings: enable/disable and poll interval (seconds)
ENABLE_CHECKPOINT_MONITOR = True
CHECKPOINT_POLL_INTERVAL = 10
DRIVE_ROOT = Path('/content/drive/MyDrive/tts_data')
DATA_ROOT = DRIVE_ROOT / VOICE_NAME
AUDIO_DIR = DATA_ROOT / 'wavs'
CSV_PATH = DATA_ROOT / 'metadata.csv'
CACHE_DIR = Path('/content/piper_cache')
CKPT_PATH = DRIVE_ROOT / 'darkman_pl.ckpt'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('CSV exists', CSV_PATH.exists())
print('Audio dir exists', AUDIO_DIR.exists())
print('Checkpoint exists', CKPT_PATH.exists())

## 3) Sanitize checkpoint and run training

In [ ]:
import sys, torch, pathlib, subprocess, os, tempfile, textwrap, threading, time, re
from pathlib import Path
# Ensure piper source is importable (editable install places src under /content/piper1-gpl/src)
PIPER_SRC = Path('/content/piper1-gpl/src')
if PIPER_SRC.exists():
    sys.path.insert(0, str(PIPER_SRC))
# allow safe globals for torch serialization in this notebook process (sanitizer run)
torch.serialization.add_safe_globals([pathlib.PosixPath])
CKPT_PATH = Path(CKPT_PATH)
if not CKPT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint not found: {CKPT_PATH}')
# sanitize checkpoint in-place to remove problematic hparams entries
ckpt = torch.load(str(CKPT_PATH), map_location='cpu', weights_only=False)
if isinstance(ckpt, dict):
    hp = ckpt.get('hyper_parameters') or ckpt.get('hparams') or {}
    def remove_model_keys(obj):
        if isinstance(obj, dict):
            for k in list(obj.keys()):
                if 'model' in k:
                    obj.pop(k, None)
                else:
                    v = obj.get(k)
                    if isinstance(v, dict):
                        remove_model_keys(v)
    if isinstance(hp, dict):
        # remove top-level dotted keys like 'model.*'
        for key in list(hp.keys()):
            if key.startswith('model.') or 'model' in key:
                hp.pop(key, None)
        # recursively remove any nested 'model' keys
        remove_model_keys(hp)
    if 'hyper_parameters' in ckpt:
        ckpt['hyper_parameters'] = hp
    else:
        ckpt['hparams'] = hp
torch.save(ckpt, str(CKPT_PATH))
print('Sanitized checkpoint inplace:', CKPT_PATH)
# Build CLI args to run piper.train inside a child Python that first registers safe globals
cli_args = [
    'fit',
    '--data.voice_name', VOICE_NAME,
    '--data.csv_path', str(DATA_ROOT / 'metadata.csv'),
    '--data.audio_dir', str(DATA_ROOT / 'wavs'),
    '--model.sample_rate', str(SAMPLE_RATE),
    '--data.espeak_voice', ESPEAK_VOICE,
    '--data.cache_dir', str(CACHE_DIR),
    '--data.config_path', str(DATA_ROOT / f"{VOICE_NAME}.json"),
    '--data.batch_size', str(BATCH_SIZE),
    '--ckpt_path', str(CKPT_PATH),
    '--weights_only', 'true',
]
# Respect MAX_EPOCHS if set (None means leave Trainer default)
if MAX_EPOCHS is not None:
    cli_args += ['--trainer.max_epochs', str(MAX_EPOCHS)]
if torch.cuda.is_available():
    cli_args += ['--trainer.accelerator', 'gpu', '--trainer.devices', '1']

# Create a small launcher script that allowlists pathlib.PosixPath then runs the piper.train module
launcher = """import runpy, sys, pathlib, torch
torch.serialization.add_safe_globals([pathlib.PosixPath])
sys.argv = ['piper.train'] + %r
runpy.run_module('piper.train', run_name='__main__')
""" % (cli_args,)

with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
    f.write(launcher)
    launcher_path = f.name

# Optional: monitor the tts_output directory for new checkpoints while training runs
outputs_dir = DATA_ROOT / 'tts_output'
def monitor_checkpoints(stop_event, interval=CHECKPOINT_POLL_INTERVAL):
    seen = set()
    while not stop_event.is_set():
        try:
            files = sorted(outputs_dir.glob('**/*.ckpt')) if outputs_dir.exists() else []
            for f in files:
                if f not in seen:
                    seen.add(f)
                    try:
                        st = f.stat()
                        print(f'[ckpt monitor] New checkpoint: {f} ({st.st_size} bytes) at {time.ctime(st.st_mtime)}')
                    except Exception as e:
                        print('[ckpt monitor] New checkpoint:', f)
        except Exception as e:
            print('Checkpoint monitor error:', e)
        time.sleep(interval)

stop_event = threading.Event()
monitor_thread = None
if ENABLE_CHECKPOINT_MONITOR:
    monitor_thread = threading.Thread(target=monitor_checkpoints, args=(stop_event,), daemon=True)
    monitor_thread.start()

python_exe = sys.executable
cmd = [python_exe, '-u', launcher_path]
print('Running:', ' '.join(map(str, cmd)))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd='/content/piper1-gpl')
last_epoch = None
epoch_re = re.compile(r'epoch[^0-9]*(\d+)', re.I)
try:
    for line in proc.stdout:
        print(line, end='')
        m = epoch_re.search(line)
        if m:
            last_epoch = int(m.group(1))
            print(f'[epoch monitor] Current epoch: {last_epoch}')
    proc.wait()
finally:
    try:
        proc.stdout.close()
    except Exception:
        pass
    if monitor_thread is not None:
        stop_event.set()
        monitor_thread.join(timeout=5)
if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, cmd)


## 4) Export trained model to ONNX

Convert the latest trained checkpoint into an optimized ONNX model and copy it to your Drive.

In [ ]:
from pathlib import Path
outputs = sorted((DATA_ROOT / 'tts_output').glob('**/*.ckpt'))
if not outputs:
    raise FileNotFoundError('No checkpoint found in tts_output')
latest_ckpt = outputs[-1]
print('Latest ckpt:', latest_ckpt)
output_model = Path('/content/model.onnx')
import subprocess, sys
subprocess.run([sys.executable, '-m', 'piper.train.export_onnx', '--checkpoint', str(latest_ckpt), '--output-file', str(output_model)], check=True)
# copy outputs to Drive
subprocess.run(['cp', str(output_model), str(DRIVE_ROOT / f'{VOICE_NAME}.onnx')], check=True)
# optional JSON metadata copy (if produced)
if (output_model.with_suffix('.onnx.json')).exists():
    subprocess.run(['cp', str(output_model.with_suffix('.onnx.json')), str(DRIVE_ROOT / f'{VOICE_NAME}.onnx.json')], check=True)
print('Export complete')
